# Brand Evaluation Demo

This notebook demonstrates how to use the brand evaluation module to test if generated images adhere to branding guidelines.

## Setup

Ensure you have GCP credentials configured:
```bash
gcloud auth application-default login
```

In [ ]:
import sys
sys.path.insert(0, '..')

from src.evaluation import (
    BrandEvaluator, 
    TestCase, 
    Guidelines, 
    GoldenDataset,
    ValidationResult,
    EvaluationReport
)

# Initialize the evaluator
evaluator = BrandEvaluator()
print("BrandEvaluator initialized successfully!")

## Evaluate a Single Image

Create a test case with your image and guidelines, then evaluate:

In [ ]:
# Define your test case
test_case = TestCase(
    id="test_001",
    image_path="path/to/your/image.png",  # Local path or gs:// URI
    original_prompt="Generate a promotional banner for summer campaign",
    guidelines=Guidelines(
        text="The image must use the primary brand color #0071CE. Logo should be visible.",
        color_palette=["#0071CE", "#FFC220", "#FFFFFF"],
        visual_style="Modern, clean, professional"
    ),
    expected_compliant=True
)

# Evaluate
result = evaluator.evaluate_image(test_case)

# Display results
print(f"Test ID: {result.test_id}")
print(f"Is Compliant: {result.is_compliant}")
print(f"Score: {result.score}/100")
print(f"Passed: {result.passed}")
print(f"\nReasoning:\n{result.reasoning}")
if result.issues:
    print(f"\nIssues: {result.issues}")

## Load and Evaluate a Golden Dataset

Load test cases from a JSON file and evaluate all at once:

In [ ]:
# Load from JSON file
dataset = evaluator.load_golden_dataset('../src/evaluation/golden_dataset.json')
print(f"Loaded {len(dataset.test_cases)} test cases")

# Show test case IDs
for tc in dataset.test_cases:
    print(f"  - {tc.id}: {tc.image_path}")

In [ ]:
# Run evaluation on all test cases
report = evaluator.evaluate_golden_dataset(dataset)

# Print summary
print("="*50)
print("EVALUATION REPORT")
print("="*50)
print(f"Total Tests:   {report.total_tests}")
print(f"Passed:        {report.passed_tests}")
print(f"Failed:        {report.failed_tests}")
print(f"Pass Rate:     {report.pass_rate:.1f}%")
print(f"Average Score: {report.average_score:.1f}/100")
print("="*50)

## View Detailed Results

In [ ]:
# Display detailed results for each test
for result in report.results:
    status = "✓ PASS" if result.passed else "✗ FAIL"
    print(f"\n[{status}] {result.test_id}")
    print(f"  Score: {result.score}/100")
    print(f"  Compliant: {result.is_compliant} (Expected: {result.expected_compliant})")
    if result.issues:
        print(f"  Issues: {result.issues}")

## Create Custom Test Cases Programmatically

In [ ]:
# Create a custom dataset
custom_dataset = GoldenDataset(
    test_cases=[
        TestCase(
            id="custom_001",
            image_path="gs://your-bucket/image1.png",
            original_prompt="Your prompt here",
            guidelines=Guidelines(
                text="Your guidelines here",
                color_palette=["#HEXCODE"],
                visual_style="Style description"
            ),
            expected_compliant=True
        ),
        # Add more test cases...
    ]
)

# Evaluate custom dataset
# report = evaluator.evaluate_golden_dataset(custom_dataset)